# TPAML 94维与696维特征数据质量检查

本Notebook检查两份特征表：

- `TPAML_Features__94*.csv`：856个样本 × 94个输入特征 + `values_ln`
- `TPAML_Features_696*.csv`：856个样本 × 696个输入特征 + `values_ln`

检查内容包括：

1. 缺失值、空字符串、非数值和无穷值；
2. 具有明确化学含义的离散计数是否为非负整数；
3. 比例、分子量、环数、共轭结构参数和实验波长是否满足基本约束；
4. 94维特征中 `Max/Min/Delta` 的内部一致性；
5. 94维与696维文件中共同且定义一致的变量是否逐行一致；
6. 完全重复的样本行；
7. 可选的统计极端值检测。

**编号说明：** 两份特征文件本身没有分子ID列，因此程序输出的“分子编号”是文件中的**行号，从1开始计数**。  
**清洗原则：** 仅自动删除确定违反硬规则的 `ERROR` 行；统计离群点、溶剂参数为0和重复行只标记为 `WARNING`，不会自动删除。

## 1. 依赖检查

In [1]:
# 若缺少依赖，自动安装到当前Jupyter内核
import importlib.util
import subprocess
import sys

required_packages = ["pandas", "numpy"]
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        print(f"正在安装 {package} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("依赖检查完成。")

依赖检查完成。


## 2. 导入库与参数设置

In [2]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Notebook与数据文件放在同一文件夹时保持为 Path(".")
DATA_DIR = Path(".")
OUTPUT_DIR = DATA_DIR / "tpaml_feature_quality_check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_ROWS = 856
EXPECTED_94_INPUT_FEATURES = 94
EXPECTED_696_INPUT_FEATURES = 696

# 是否执行基于MAD的统计极端值检测。
# 统计离群点不等于化学错误，因此只标记为WARNING。
CHECK_STATISTICAL_OUTLIERS = True
ROBUST_Z_THRESHOLD = 15.0

# 是否在屏幕上逐条打印统计离群点。
# 默认False，避免数百条输出刷屏；所有离群点仍会保存到CSV。
PRINT_STATISTICAL_OUTLIERS = False

# 清洗时只删除ERROR，默认不删除WARNING。
DROP_WARNING_ROWS = False

print("当前工作目录：", Path.cwd())
print("输出目录：", OUTPUT_DIR.resolve())

当前工作目录： c:\Users\pc\Desktop\TPACS
输出目录： C:\Users\pc\Desktop\TPACS\tpaml_feature_quality_check


## 3. 自动查找并读取94维和696维特征文件

In [3]:
def find_feature_file(data_dir: Path, feature_number: int) -> Path:
    """自动查找CSV或Excel格式的特征文件。"""
    patterns = [
        f"TPAML_Features_{'_' if feature_number == 94 else ''}{feature_number}.csv",
        f"TPAML_Features_{'_' if feature_number == 94 else ''}{feature_number}*.csv",
        f"*Features*{feature_number}*.csv",
        f"TPAML_Features_{'_' if feature_number == 94 else ''}{feature_number}.xlsx",
        f"TPAML_Features_{'_' if feature_number == 94 else ''}{feature_number}*.xlsx",
        f"*Features*{feature_number}*.xlsx",
    ]

    candidates = []
    for pattern in patterns:
        candidates.extend(data_dir.glob(pattern))

    # 去重，排除本Notebook产生的结果文件
    unique = []
    seen = set()
    for path in candidates:
        if path.name.startswith("cleaned_") or "quality" in path.name.lower():
            continue
        resolved = path.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(path)

    if not unique:
        raise FileNotFoundError(
            f"在 {data_dir.resolve()} 中找不到包含 {feature_number} 的特征CSV/Excel文件。"
        )

    # 优先选择不带括号后缀的标准文件名，否则选名称最短者
    unique.sort(key=lambda p: ("(" in p.name, len(p.name), p.name))
    return unique[0]


def read_feature_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"不支持的文件格式：{path.suffix}")


FILE_94 = find_feature_file(DATA_DIR, 94)
FILE_696 = find_feature_file(DATA_DIR, 696)

df94_raw = read_feature_table(FILE_94)
df696_raw = read_feature_table(FILE_696)

print("94维文件：", FILE_94.resolve())
print("696维文件：", FILE_696.resolve())
print("94维表形状：", df94_raw.shape)
print("696维表形状：", df696_raw.shape)

display(df94_raw.head(3))
display(df696_raw.head(3))

94维文件： C:\Users\pc\Desktop\TPACS\TPAML_Features__94.csv
696维文件： C:\Users\pc\Desktop\TPACS\TPAML_Features_696.csv
94维表形状： (856, 95)
696维表形状： (856, 697)


,Num-Conju-Stru,Conju-Num-Atom-All,Conju-Num-Atom-Ratio,Conju-Wt-Ave,Full-Mol Wiener Index,Conju-Num-Atom-Individual,Conju-Wt-Part,Conju-Wt-Ave.1,Conju-Max-Distance,Conju-Branch-Ratio,...,Atomic-MR-NegativeDisCoef,Atomic-MR-NegativeDisCoef-PairMean,Atomic-MR-GradSum,Atomic-MR-GradSum-PairMean,Atomic-MR-MaxMinDisRatio,Atomic-MR-LaplaceSum,Atomic-MR-Laplace-PairMean,ET(30) (Solvent),Wavelength (Exp nm),values_ln
0,3,28,0.848485,0.796270,2.623106,16,200.156,12.509750,10,0.200000,...,479.823593,3.998530,51.833220,0.431943,0.600000,29.820911,0.248508,37.4,750,3.871201
1,3,29,0.852941,0.806646,2.694296,17,219.154,12.891412,11,0.181818,...,493.552675,3.629064,61.560809,0.452653,0.545455,34.391462,0.252878,37.4,760,3.828641
2,3,29,0.852941,0.813462,2.694296,17,235.609,13.859353,11,0.181818,...,565.325797,4.156807,63.226387,0.464900,0.545455,35.120000,0.258235,37.4,750,4.477337


,ccc,cccc(c)N,cc(c)N(CC)CC,cc(c)O,CCN,cc(c)C,c1ccccc1,cccc(c)C,cccc(c)[N+],CCN(C)c,...,Conju-Elec-Distance-Coef,Conju-Elec-Distance-Coef-Norm,Conju-MultiElec-Distance-Coef,Conju-MultiElec-Distance-Coef-Norm,Conju-Stru-VSA,ET(30) (Solvent),Dielectic Constant (Solvent),Dipole Moment (Solvent),Wavelength (Exp nm),values_ln
0,24,4,1,1,2,1,4,2,2,2,...,6.154858,3.925000,0.0,0.0,5.458523,37.4,7.6,1.75,750,3.871201
1,24,4,1,1,2,1,4,2,2,2,...,6.493754,4.860294,0.0,0.0,5.485579,37.4,7.6,1.75,760,3.828641
2,24,4,1,1,2,1,4,2,2,2,...,6.493754,4.860294,0.0,0.0,5.559745,37.4,7.6,1.75,750,4.477337


## 4. 数据质量检查函数

In [4]:
ISSUE_COLUMNS = [
    "dataset",
    "molecule_no",
    "field",
    "issue_type",
    "severity",
    "observed_value",
    "expected_rule",
]

def new_issue(
    dataset: str,
    molecule_no,
    field: str,
    issue_type: str,
    severity: str,
    observed_value,
    expected_rule: str,
) -> dict:
    return {
        "dataset": dataset,
        "molecule_no": molecule_no,
        "field": field,
        "issue_type": issue_type,
        "severity": severity,
        "observed_value": observed_value,
        "expected_rule": expected_rule,
    }


def numeric_copy_with_parse_issues(
    df: pd.DataFrame,
    dataset_name: str,
    issues: list[dict],
) -> pd.DataFrame:
    """
    将所有列转换为数值。
    转换前非空但转换后为NaN的单元格，被记录为非数值异常。
    """
    converted_columns = {}

    for column in df.columns:
        original = df[column]

        blank_mask = original.isna() | original.astype(str).str.strip().eq("")
        for row_index in np.flatnonzero(blank_mask.to_numpy()):
            issues.append(new_issue(
                dataset_name,
                int(row_index) + 1,
                column,
                "数据缺失",
                "ERROR",
                original.iloc[row_index],
                "该特征不得缺失",
            ))

        converted = pd.to_numeric(original, errors="coerce")
        parse_error_mask = (~blank_mask) & converted.isna()

        for row_index in np.flatnonzero(parse_error_mask.to_numpy()):
            issues.append(new_issue(
                dataset_name,
                int(row_index) + 1,
                column,
                "非数值或无法解析",
                "ERROR",
                original.iloc[row_index],
                "特征表中的单元格应为数值",
            ))

        nonfinite_mask = converted.notna() & ~np.isfinite(converted)
        for row_index in np.flatnonzero(nonfinite_mask.to_numpy()):
            issues.append(new_issue(
                dataset_name,
                int(row_index) + 1,
                column,
                "无穷大或非有限数值",
                "ERROR",
                original.iloc[row_index],
                "必须为有限数值",
            ))

        converted_columns[column] = converted

    return pd.DataFrame(converted_columns, index=df.index)


def add_mask_issues(
    issues: list[dict],
    df: pd.DataFrame,
    mask: pd.Series,
    dataset_name: str,
    field: str,
    issue_type: str,
    severity: str,
    expected_rule: str,
) -> None:
    mask = mask.fillna(False)
    for row_index in np.flatnonzero(mask.to_numpy()):
        if field in df.columns:
            value = df.loc[row_index, field]
        else:
            value = ""
        issues.append(new_issue(
            dataset_name,
            int(row_index) + 1,
            field,
            issue_type,
            severity,
            value,
            expected_rule,
        ))


def check_nonnegative_integer(
    df: pd.DataFrame,
    columns: list[str],
    dataset_name: str,
    issues: list[dict],
) -> None:
    for column in columns:
        if column not in df.columns:
            continue
        series = df[column]
        mask = series.notna() & (
            (series < 0) |
            (~np.isclose(series, np.round(series), atol=1e-8))
        )
        add_mask_issues(
            issues, df, mask, dataset_name, column,
            "计数或离散结构特征不是非负整数",
            "ERROR",
            "该特征应为0或正整数",
        )


def check_positive(
    df: pd.DataFrame,
    columns: list[str],
    dataset_name: str,
    issues: list[dict],
) -> None:
    for column in columns:
        if column not in df.columns:
            continue
        mask = df[column].notna() & (df[column] <= 0)
        add_mask_issues(
            issues, df, mask, dataset_name, column,
            "物理量不是正数",
            "ERROR",
            "该特征应大于0",
        )


def check_closed_interval(
    df: pd.DataFrame,
    columns: list[str],
    lower: float,
    upper: float,
    dataset_name: str,
    issues: list[dict],
) -> None:
    for column in columns:
        if column not in df.columns:
            continue
        mask = df[column].notna() & (
            (df[column] < lower) | (df[column] > upper)
        )
        add_mask_issues(
            issues, df, mask, dataset_name, column,
            f"数值超出[{lower}, {upper}]",
            "ERROR",
            f"{lower} ≤ 数值 ≤ {upper}",
        )


def robust_outlier_issues(
    df: pd.DataFrame,
    columns: list[str],
    dataset_name: str,
    threshold: float,
) -> list[dict]:
    """
    使用MAD计算稳健Z分数。仅作为人工复核提醒，不自动删除。
    """
    outlier_issues = []

    for column in columns:
        if column not in df.columns:
            continue

        series = df[column].dropna()
        if len(series) < 50 or series.nunique() <= 10:
            continue

        median = float(series.median())
        mad = float(np.median(np.abs(series.to_numpy() - median)))

        if not np.isfinite(mad) or mad == 0:
            continue

        robust_z = 0.6744897501960817 * (df[column] - median) / mad
        mask = robust_z.abs() > threshold

        for row_index in np.flatnonzero(mask.fillna(False).to_numpy()):
            outlier_issues.append(new_issue(
                dataset_name,
                int(row_index) + 1,
                column,
                "统计极端值",
                "WARNING",
                df.loc[row_index, column],
                f"|robust z| > {threshold}；仅提示人工核查，不代表化学错误",
            ))

    return outlier_issues

## 5. 执行确定性缺失与化学/数学合理性检查

In [5]:
def check_feature_dataset(
    raw_df: pd.DataFrame,
    dataset_name: str,
    expected_input_features: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    issues: list[dict] = []
    schema_issues: list[dict] = []

    # ---------- 表结构检查 ----------
    expected_total_columns = expected_input_features + 1  # + values_ln

    if len(raw_df) != EXPECTED_ROWS:
        schema_issues.append({
            "dataset": dataset_name,
            "schema_issue": "样本行数与论文建模数据不一致",
            "observed": len(raw_df),
            "expected": EXPECTED_ROWS,
        })

    if raw_df.shape[1] != expected_total_columns:
        schema_issues.append({
            "dataset": dataset_name,
            "schema_issue": "列数不一致",
            "observed": raw_df.shape[1],
            "expected": expected_total_columns,
        })

    if "values_ln" not in raw_df.columns:
        schema_issues.append({
            "dataset": dataset_name,
            "schema_issue": "缺少目标变量列 values_ln",
            "observed": "不存在",
            "expected": "必须存在",
        })

    # pandas读取重复表头时常自动生成 .1、.2 后缀，因此单独提示
    mangled_columns = [
        column for column in raw_df.columns
        if re.search(r"\.\d+$", str(column))
    ]
    for column in mangled_columns:
        schema_issues.append({
            "dataset": dataset_name,
            "schema_issue": "发现可能由重复表头产生的列名后缀",
            "observed": column,
            "expected": "核查原始表头及该列定义",
        })

    df = numeric_copy_with_parse_issues(raw_df, dataset_name, issues)

    # ---------- 重复样本 ----------
    duplicate_mask = df.duplicated(keep=False)
    for row_index in np.flatnonzero(duplicate_mask.to_numpy()):
        issues.append(new_issue(
            dataset_name,
            int(row_index) + 1,
            "ALL_FEATURES",
            "整行数据与其他样本完全重复",
            "WARNING",
            "duplicate row",
            "可能是相同实验记录，也可能是重复样本；需结合原始分子ID核查",
        ))

    # ---------- 公共硬规则 ----------
    check_positive(
        df,
        [
            "MolWt",
            "HeavyAtomMolWt",
            "ExactMolWt",
            "MolMR",
            "Conju-Wt-Part",
            "Conju-Max-Distance",
            "Conju-Stru-VSA",
            "Full-Mol Wiener Index",
            "Conju-Stru Wiener Index",
            "Conju-Stru-Wiener-Index",
            "Conju-Num-Atom-All",
            "Conju-Num-Atom-Individual",
        ],
        dataset_name,
        issues,
    )

    # 比例型变量：仅检查在论文/SI中定义明确且文件中确实按比例存储的列
    check_closed_interval(
        df,
        ["FractionCSP3", "Conju-Num-Atom-Ratio", "Conju-Wt-Ave"],
        0.0,
        1.0,
        dataset_name,
        issues,
    )

    # 论文建模数据明确限制为600–1100 nm
    if "Wavelength (Exp nm)" in df.columns:
        wavelength_mask = df["Wavelength (Exp nm)"].notna() & (
            (df["Wavelength (Exp nm)"] < 600) |
            (df["Wavelength (Exp nm)"] > 1100)
        )
        add_mask_issues(
            issues, df, wavelength_mask, dataset_name,
            "Wavelength (Exp nm)",
            "测试波长超出论文建模范围",
            "ERROR",
            "600 nm ≤ Wavelength ≤ 1100 nm",
        )

    # 溶剂参数为0更可能是未知值编码，不直接判定为错误
    # ET(30)=0或介电常数=0通常表示未知值编码。
    # 偶极矩为0对苯、环己烷等对称非极性溶剂可能是合理值，因此不作为缺失。
    solvent_zero_columns = [
        "ET(30) (Solvent)",
        "Dielectic Constant (Solvent)",
    ]
    for column in solvent_zero_columns:
        if column in df.columns:
            zero_mask = df[column].notna() & np.isclose(df[column], 0.0)
            add_mask_issues(
                issues, df, zero_mask, dataset_name, column,
                "溶剂参数为0，可能代表未知或缺失溶剂信息",
                "WARNING",
                "返回原始溶剂名称核查；不自动删除",
            )

    # 保守的溶剂物性范围，仅作为明显录入错误检查
    if "ET(30) (Solvent)" in df.columns:
        mask = df["ET(30) (Solvent)"].notna() & (
            (df["ET(30) (Solvent)"] < 0) |
            (df["ET(30) (Solvent)"] > 80)
        )
        add_mask_issues(
            issues, df, mask, dataset_name, "ET(30) (Solvent)",
            "ET(30)超出保守合理范围", "ERROR", "0–80",
        )

    if "Dielectic Constant (Solvent)" in df.columns:
        mask = df["Dielectic Constant (Solvent)"].notna() & (
            (df["Dielectic Constant (Solvent)"] < 0) |
            (df["Dielectic Constant (Solvent)"] > 200)
        )
        add_mask_issues(
            issues, df, mask, dataset_name, "Dielectic Constant (Solvent)",
            "溶剂介电常数超出保守合理范围", "ERROR", "0–200",
        )

    if "Dipole Moment (Solvent)" in df.columns:
        mask = df["Dipole Moment (Solvent)"].notna() & (
            (df["Dipole Moment (Solvent)"] < 0) |
            (df["Dipole Moment (Solvent)"] > 20)
        )
        add_mask_issues(
            issues, df, mask, dataset_name, "Dipole Moment (Solvent)",
            "溶剂偶极矩超出保守合理范围", "ERROR", "0–20 D",
        )

    # ---------- 696维专属规则 ----------
    if expected_input_features == 696:
        # 696维表的前564列为MFF片段出现次数，应为非负整数
        if "MaxEStateIndex" in df.columns:
            mff_end = df.columns.get_loc("MaxEStateIndex")
            mff_columns = list(df.columns[:mff_end])
        else:
            mff_columns = list(df.columns[:564])

        check_nonnegative_integer(
            df,
            mff_columns,
            dataset_name,
            issues,
        )

        explicit_count_columns = []
        for column in df.columns:
            name = str(column)
            if (
                name.startswith("Num")
                or name.endswith("Count")
                or name in {
                    "RingCount",
                    "NHOHCount",
                    "NOCount",
                    "HeavyAtomCount",
                    "Conju-Num-Atom-Individual",
                    "Conju-Max-Distance",
                    "Conju-Branch-Num",
                    "Conju-sp2N-Num",
                    "Apperent Conju-Electron Count ",
                }
            ):
                explicit_count_columns.append(column)

        check_nonnegative_integer(
            df,
            explicit_count_columns,
            dataset_name,
            issues,
        )

        # RDKit分子量关系
        if {"HeavyAtomMolWt", "MolWt"}.issubset(df.columns):
            mask = (
                df["HeavyAtomMolWt"].notna()
                & df["MolWt"].notna()
                & (df["HeavyAtomMolWt"] > df["MolWt"] + 1e-8)
            )
            add_mask_issues(
                issues, df, mask, dataset_name,
                "HeavyAtomMolWt",
                "重原子分子量大于总分子量",
                "ERROR",
                "HeavyAtomMolWt ≤ MolWt",
            )

        # 电荷最大/最小关系
        if {"MinPartialCharge", "MaxPartialCharge"}.issubset(df.columns):
            mask = df["MinPartialCharge"] > df["MaxPartialCharge"]
            add_mask_issues(
                issues, df, mask, dataset_name,
                "MinPartialCharge/MaxPartialCharge",
                "最小部分电荷大于最大部分电荷",
                "ERROR",
                "MinPartialCharge ≤ MaxPartialCharge",
            )

        if {"MinAbsPartialCharge", "MaxAbsPartialCharge"}.issubset(df.columns):
            mask = df["MinAbsPartialCharge"] > df["MaxAbsPartialCharge"]
            add_mask_issues(
                issues, df, mask, dataset_name,
                "MinAbsPartialCharge/MaxAbsPartialCharge",
                "最小绝对部分电荷大于最大绝对部分电荷",
                "ERROR",
                "MinAbsPartialCharge ≤ MaxAbsPartialCharge",
            )

        # 子类环数不应超过总环数
        if "RingCount" in df.columns:
            for column in [
                "NumAromaticRings",
                "NumAliphaticRings",
                "NumSaturatedRings",
            ]:
                if column in df.columns:
                    mask = df[column] > df["RingCount"]
                    add_mask_issues(
                        issues, df, mask, dataset_name,
                        column,
                        "子类环数大于总环数",
                        "ERROR",
                        f"{column} ≤ RingCount",
                    )

    # ---------- 94维专属规则 ----------
    if expected_input_features == 94:
        count_columns_94 = [
            "Num-Conju-Stru",
            "Conju-Num-Atom-All",
            "Conju-Num-Atom-Individual",
            "Conju-Max-Distance",
            "Apperant-Elec-Count-Sum",
            "Apperant-Elec-Count-Max",
            "Apperant-Elec-Count-Min",
            "Apperant-Elec-Count-Delta",
        ]
        check_nonnegative_integer(
            df,
            count_columns_94,
            dataset_name,
            issues,
        )

        # 五组MFF-MOE统计量必须满足Max >= Min且Delta = Max-Min
        groups = [
            "Apperant-Elec-Count",
            "PEOE-Charge",
            "EState-Indice",
            "Atomic-LogP",
            "Atomic-MR",
        ]
        for group in groups:
            max_col = f"{group}-Max"
            min_col = f"{group}-Min"
            delta_col = f"{group}-Delta"

            if {max_col, min_col, delta_col}.issubset(df.columns):
                max_min_mask = df[max_col] < df[min_col]
                add_mask_issues(
                    issues, df, max_min_mask, dataset_name,
                    f"{max_col}/{min_col}",
                    "最大值小于最小值",
                    "ERROR",
                    f"{max_col} ≥ {min_col}",
                )

                delta_error = (
                    df[max_col] - df[min_col] - df[delta_col]
                ).abs() > 1e-6
                add_mask_issues(
                    issues, df, delta_error, dataset_name,
                    delta_col,
                    "Delta与Max-Min不一致",
                    "ERROR",
                    f"{delta_col} = {max_col} - {min_col}",
                )

    issue_df = pd.DataFrame(issues, columns=ISSUE_COLUMNS)
    schema_df = pd.DataFrame(schema_issues)

    return df, issue_df, schema_df


df94, issues94, schema94 = check_feature_dataset(
    df94_raw,
    "94_features",
    EXPECTED_94_INPUT_FEATURES,
)

df696, issues696, schema696 = check_feature_dataset(
    df696_raw,
    "696_features",
    EXPECTED_696_INPUT_FEATURES,
)

print("确定性检查完成。")
print("94维问题记录数：", len(issues94))
print("696维问题记录数：", len(issues696))

确定性检查完成。
94维问题记录数： 17
696维问题记录数： 26


## 6. 检查94维与696维文件的逐行一致性

In [6]:
cross_file_issues = []

if len(df94) != len(df696):
    cross_file_issues.append(new_issue(
        "CROSS_FILE",
        np.nan,
        "row_count",
        "两份文件样本行数不一致",
        "ERROR",
        f"94维={len(df94)}, 696维={len(df696)}",
        "两份文件应对应同一批856个样本",
    ))
else:
    # 仅比较在两份文件中定义和数值尺度明确一致的字段。
    # Conju-Stru-VSA在两套特征器中的尺度不同，
    # Conju-Num-Atom-Individual的构建方式也有差异，因此不做强制比较。
    stable_shared_columns = [
        "Num-Conju-Stru",
        "Conju-Num-Atom-Ratio",
        "Conju-Wt-Ave",
        "Full-Mol Wiener Index",
        "Conju-Wt-Part",
        "Conju-Max-Distance",
        "Conju-Branch-Ratio",
        "ET(30) (Solvent)",
        "Wavelength (Exp nm)",
        "values_ln",
    ]

    for column in stable_shared_columns:
        if column not in df94.columns or column not in df696.columns:
            continue

        equal_mask = np.isclose(
            df94[column].to_numpy(dtype=float),
            df696[column].to_numpy(dtype=float),
            rtol=1e-7,
            atol=1e-9,
            equal_nan=True,
        )
        mismatch_indices = np.flatnonzero(~equal_mask)

        for row_index in mismatch_indices:
            cross_file_issues.append(new_issue(
                "CROSS_FILE",
                int(row_index) + 1,
                column,
                "94维与696维文件中的共同字段不一致",
                "WARNING",
                f"94维={df94.loc[row_index, column]}, 696维={df696.loc[row_index, column]}",
                "核查两套特征器版本和原始输入",
            ))

cross_file_df = pd.DataFrame(cross_file_issues, columns=ISSUE_COLUMNS)

print("跨文件一致性问题数：", len(cross_file_df))
if not cross_file_df.empty:
    display(cross_file_df.head(20))

跨文件一致性问题数： 0


## 7. 可选：统计极端值检测

In [7]:
outlier_issues = []

if CHECK_STATISTICAL_OUTLIERS:
    # 94维：检查所有输入特征，不含目标值
    columns94_for_outliers = [
        column for column in df94.columns
        if column != "values_ln"
    ]

    # 696维：排除前564个高维稀疏MFF计数，只检查RDKit、共轭和实验条件特征
    if "MaxEStateIndex" in df696.columns:
        descriptor_start = df696.columns.get_loc("MaxEStateIndex")
        columns696_for_outliers = [
            column for column in df696.columns[descriptor_start:]
            if column != "values_ln"
        ]
    else:
        columns696_for_outliers = [
            column for column in df696.columns[564:]
            if column != "values_ln"
        ]

    outlier_issues.extend(robust_outlier_issues(
        df94,
        columns94_for_outliers,
        "94_features",
        ROBUST_Z_THRESHOLD,
    ))
    outlier_issues.extend(robust_outlier_issues(
        df696,
        columns696_for_outliers,
        "696_features",
        ROBUST_Z_THRESHOLD,
    ))

outlier_df = pd.DataFrame(outlier_issues, columns=ISSUE_COLUMNS)

print("统计极端值记录数：", len(outlier_df))
print(
    "说明：统计极端值只表示数值相对罕见，不等于化学错误，"
    "不会在默认清洗过程中删除。"
)

if PRINT_STATISTICAL_OUTLIERS and not outlier_df.empty:
    display(outlier_df)

统计极端值记录数： 543
说明：统计极端值只表示数值相对罕见，不等于化学错误，不会在默认清洗过程中删除。


## 8. 汇总并打印有问题的分子编号及字段名称

In [8]:
all_issue_frames = [
    issues94,
    issues696,
    cross_file_df,
]

if CHECK_STATISTICAL_OUTLIERS:
    all_issue_frames.append(outlier_df)

all_issues = pd.concat(all_issue_frames, ignore_index=True)
all_issues = all_issues.drop_duplicates().sort_values(
    ["severity", "dataset", "molecule_no", "field"],
    na_position="last",
).reset_index(drop=True)

# 确定性问题：不包含统计极端值
deterministic_issues = all_issues[
    all_issues["issue_type"] != "统计极端值"
].copy()

missing_issues = deterministic_issues[
    deterministic_issues["issue_type"] == "数据缺失"
].copy()

hard_errors = deterministic_issues[
    deterministic_issues["severity"] == "ERROR"
].copy()

warnings = deterministic_issues[
    deterministic_issues["severity"] == "WARNING"
].copy()

print("=" * 90)
print("数据质量检查结果")
print("=" * 90)
print(f"缺失项数量：{len(missing_issues)}")
print(f"确定性ERROR数量：{len(hard_errors)}")
print(f"确定性WARNING数量：{len(warnings)}")
print(f"统计极端值数量：{len(outlier_df)}")
print()

def print_issue_lines(issue_table: pd.DataFrame, title: str) -> None:
    print(title)
    print("-" * 90)

    if issue_table.empty:
        print("未发现。")
        print()
        return

    for _, row in issue_table.iterrows():
        molecule_text = (
            "表结构"
            if pd.isna(row["molecule_no"])
            else f"分子编号 {int(row['molecule_no'])}"
        )
        print(
            f"[{row['dataset']}] {molecule_text} | "
            f"字段：{row['field']} | "
            f"{row['severity']} | "
            f"{row['issue_type']} | "
            f"观测值：{row['observed_value']}"
        )
    print()

print_issue_lines(missing_issues, "一、存在缺失项的分子")
print_issue_lines(hard_errors, "二、违反硬性化学或数学规则的分子")
print_issue_lines(warnings, "三、需要人工核查但不自动删除的分子")

if PRINT_STATISTICAL_OUTLIERS:
    print_issue_lines(outlier_df, "四、统计极端值")

display(all_issues.head(50))

数据质量检查结果
缺失项数量：0
确定性ERROR数量：0
确定性WARNING数量：43
统计极端值数量：543

一、存在缺失项的分子
------------------------------------------------------------------------------------------
未发现。

二、违反硬性化学或数学规则的分子
------------------------------------------------------------------------------------------
未发现。

三、需要人工核查但不自动删除的分子
------------------------------------------------------------------------------------------
[696_features] 分子编号 113 | 字段：Dielectic Constant (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 113 | 字段：ET(30) (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 114 | 字段：Dielectic Constant (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 114 | 字段：ET(30) (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 115 | 字段：Dielectic Constant (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 115 | 字段：ET(30) (Solvent) | WARNING | 溶剂参数为0，可能代表未知或缺失溶剂信息 | 观测值：0.0
[696_features] 分子编号 116 | 字段：Dielectic C

,dataset,molecule_no,field,issue_type,severity,observed_value,expected_rule
0,696_features,16,VSA_EState3,统计极端值,WARNING,38.297276,|robust z| > 15.0；仅提示人工核查，不代表化学错误
1,696_features,39,EState_VSA5,统计极端值,WARNING,436.61587,|robust z| > 15.0；仅提示人工核查，不代表化学错误
2,696_features,39,SMR_VSA5,统计极端值,WARNING,569.055627,|robust z| > 15.0；仅提示人工核查，不代表化学错误
3,696_features,39,VSA_EState1,统计极端值,WARNING,53.878241,|robust z| > 15.0；仅提示人工核查，不代表化学错误
4,696_features,40,EState_VSA5,统计极端值,WARNING,642.082162,|robust z| > 15.0；仅提示人工核查，不代表化学错误
5,696_features,40,Kappa3,统计极端值,WARNING,93.735522,|robust z| > 15.0；仅提示人工核查，不代表化学错误
6,696_features,40,NumRotatableBonds,统计极端值,WARNING,137.0,|robust z| > 15.0；仅提示人工核查，不代表化学错误
7,696_features,40,SMR_VSA5,统计极端值,WARNING,774.521919,|robust z| > 15.0；仅提示人工核查，不代表化学错误
8,696_features,40,SlogP_VSA5,统计极端值,WARNING,819.551437,|robust z| > 15.0；仅提示人工核查，不代表化学错误
9,696_features,40,VSA_EState1,统计极端值,WARNING,54.726932,|robust z| > 15.0；仅提示人工核查，不代表化学错误


## 9. 按分子汇总问题

In [9]:
molecule_issue_summary = (
    all_issues.dropna(subset=["molecule_no"])
    .groupby(["dataset", "molecule_no"], as_index=False)
    .agg(
        error_count=("severity", lambda x: int((x == "ERROR").sum())),
        warning_count=("severity", lambda x: int((x == "WARNING").sum())),
        problem_fields=("field", lambda x: "; ".join(sorted(set(map(str, x))))),
        issue_types=("issue_type", lambda x: "; ".join(sorted(set(map(str, x))))),
    )
    .sort_values(
        ["error_count", "warning_count", "dataset", "molecule_no"],
        ascending=[False, False, True, True],
    )
)

display(molecule_issue_summary.head(100))

,dataset,molecule_no,error_count,warning_count,problem_fields,issue_types
221,696_features,808,0,33,BertzCT; Chi0; Chi0n; Chi0v; Chi1; Chi1n; Chi1...,统计极端值
218,696_features,804,0,19,BertzCT; Chi0n; Chi0v; Chi1n; Chi1v; Chi2n; Ch...,统计极端值
310,94_features,803,0,15,Apperant-Elec-Count-NegativeDisCoef; Apperant-...,统计极端值
311,94_features,804,0,14,Apperant-Elec-Count-NegativeDisCoef; Apperant-...,统计极端值
314,94_features,808,0,14,Apperant-Elec-Count-NegativeDisCoef; Apperant-...,统计极端值
...,...,...,...,...,...,...
16,696_features,83,0,1,VSA_EState3,统计极端值
17,696_features,84,0,1,VSA_EState3,统计极端值
18,696_features,90,0,1,VSA_EState5,统计极端值
19,696_features,91,0,1,VSA_EState5,统计极端值


## 10. 生成清洗后的数据并导出报告

In [10]:
def rows_to_drop(
    issue_table: pd.DataFrame,
    dataset_name: str,
    drop_warnings: bool,
) -> list[int]:
    subset = issue_table[
        (issue_table["dataset"] == dataset_name)
        & issue_table["molecule_no"].notna()
    ].copy()

    if not drop_warnings:
        subset = subset[subset["severity"] == "ERROR"]

    # molecule_no为1-based，DataFrame索引为0-based
    return sorted({
        int(number) - 1
        for number in subset["molecule_no"]
    })


drop94 = rows_to_drop(all_issues, "94_features", DROP_WARNING_ROWS)
drop696 = rows_to_drop(all_issues, "696_features", DROP_WARNING_ROWS)

cleaned94 = df94_raw.drop(index=drop94).reset_index(drop=True)
cleaned696 = df696_raw.drop(index=drop696).reset_index(drop=True)

# 保存完整问题报告
all_issues.to_csv(
    OUTPUT_DIR / "TPAML_94_696_quality_issues.csv",
    index=False,
    encoding="utf-8-sig",
)

deterministic_issues.to_csv(
    OUTPUT_DIR / "TPAML_94_696_deterministic_issues.csv",
    index=False,
    encoding="utf-8-sig",
)

outlier_df.to_csv(
    OUTPUT_DIR / "TPAML_94_696_statistical_outliers.csv",
    index=False,
    encoding="utf-8-sig",
)

molecule_issue_summary.to_csv(
    OUTPUT_DIR / "TPAML_94_696_molecule_issue_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

schema_summary = pd.concat(
    [schema94, schema696],
    ignore_index=True,
)
schema_summary.to_csv(
    OUTPUT_DIR / "TPAML_94_696_schema_issues.csv",
    index=False,
    encoding="utf-8-sig",
)

cleaned94.to_csv(
    OUTPUT_DIR / "cleaned_TPAML_Features_94.csv",
    index=False,
    encoding="utf-8-sig",
)

cleaned696.to_csv(
    OUTPUT_DIR / "cleaned_TPAML_Features_696.csv",
    index=False,
    encoding="utf-8-sig",
)

# 另存一份便于直接阅读的文本列表
with (OUTPUT_DIR / "problem_molecules_and_fields.txt").open(
    "w",
    encoding="utf-8",
) as handle:
    for _, row in deterministic_issues.iterrows():
        molecule_text = (
            "表结构"
            if pd.isna(row["molecule_no"])
            else f"分子编号 {int(row['molecule_no'])}"
        )
        handle.write(
            f"[{row['dataset']}] {molecule_text} | "
            f"字段：{row['field']} | "
            f"{row['severity']} | {row['issue_type']} | "
            f"观测值：{row['observed_value']}\n"
        )

print("导出完成：", OUTPUT_DIR.resolve())
print("94维删除的ERROR行数：", len(drop94))
print("696维删除的ERROR行数：", len(drop696))
print("清洗后94维形状：", cleaned94.shape)
print("清洗后696维形状：", cleaned696.shape)

print("\n生成文件：")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(" -", path.name)

导出完成： C:\Users\pc\Desktop\TPACS\tpaml_feature_quality_check
94维删除的ERROR行数： 0
696维删除的ERROR行数： 0
清洗后94维形状： (856, 95)
清洗后696维形状： (856, 697)

生成文件：
 - cleaned_TPAML_Features_696.csv
 - cleaned_TPAML_Features_94.csv
 - problem_molecules_and_fields.txt
 - TPAML_94_696_deterministic_issues.csv
 - TPAML_94_696_molecule_issue_summary.csv
 - TPAML_94_696_quality_issues.csv
 - TPAML_94_696_schema_issues.csv
 - TPAML_94_696_statistical_outliers.csv


## 结果解释

- `ERROR`：明确缺失、非数值、无穷值，或违反基本化学/数学硬规则。默认会从清洗版数据中删除对应行。
- `WARNING`：可能需要返回原始文献或分子结构核查，但不能仅凭当前特征表断言错误。例如：
  - ET(30)或介电常数为0；
  - 两条特征记录完全相同；
  - 统计极端值；
  - 两套特征器中共同字段存在差异。
- `统计极端值`：只反映相对于数据分布较罕见，可能恰好对应大型共轭分子或高性能分子，因此**不会自动删除**。

输出文件位于当前目录下的 `tpaml_feature_quality_check` 文件夹。